In [ ]:
import os, sys
path_to_scr_folder=os.path.join(os.path.dirname(os.path.abspath('')), 'src')
sys.path.append(path_to_scr_folder)

In [ ]:
import ipywidgets as widgets
from IPython.display import display
from ipyfilechooser import FileChooser
from pathlib import Path
from interface_jupyter import Interface
import ipywidgets as widgets
import subprocess
import sys
import time
import ipywidgets as widgets
from IPython.display import display



class GCGCMSAnalysisUI(Interface):
    """
     GC×GC-MS Analysis UI with improved error handling and flexible file/folder selection.
    Provides a widget-based interface for configuring and running GCGCMS analysis.
    Users can select individual files, folders, or subfolders - all compatible files will be processed.
    """
    
    def __init__(self):
        """Initialize the GCMS Analysis UI with default parameters and widgets."""
        super().__init__(supported_extensions=('.h5', '.cdf')) # respecter la logique  de priorite, le 1er de la liste = celui qui sera privilegie lors du choix entre les 2 extentions
        self._setup_default_parameters()
        self._create_parameter_widgets()
        self._create_gcgcms_widgets()
        self._create_base_widgets()
        self._create_action_widgets()
        self._setup_callbacks()
        self._setup_environment()

        
    def _setup_default_parameters(self):
        """Initialize default analysis parameters."""
        # Public parameters (configurable via UI)
        self.abs_threshold = "1000"
        self.rel_threshold = "0.001"
        self.noise_factor = "5"
        self.mod_time = ""
        self.sigma_ratio = "1.5"
        self.min_sigma = "1"
        self.max_sigma = "3"
        self.nist = False
        self.plot = True
        self.is_area_deconvolution = False

        # Private parameters (fixed for this UI)
        self._min_distance = 1
        self._num_sigma = 10
        self._overlap = 0.5
        self._match_factor_min = 600
        self._cluster = True
        self._min_samples = 1
        self._eps = 0.001
        self._w_mod_time = "0"
        self._formated_spectra = True
        self._min_persistence = "0.0002"
        
    
    def _create_parameter_widgets(self):
        """Create parameter input widgets."""
        self.w_noise_factor = widgets.Text(value=self.noise_factor)
        self.noise_factor = self._bold_widget("Noise factor", self.w_noise_factor)
        self.noise_factor_def = self.create_help_text(
            "Noise scaling factor used to filter detected peaks."
            "A peak is retained if its intensity is greater than the maximum intensity multiplied by this factor."
        )
        # self.w_min_persistence = widgets.Text(value=self.min_persistence)
        # self.min_persistence = self._bold_widget("Minimum persistence", self.w_min_persistence)
        # self.min_persistence_def = self.create_help_text(
        #     "Minimum topological persistence threshold that a peak must exceed to be considered a true signal rather than noise."
        # )
        self.w_abs_threshold = widgets.Text(value=self.abs_threshold)
        self.abs_threshold = self._bold_widget("Absolute threshold", self.w_abs_threshold)
        self.abs_threshold_def = self.create_help_text(
            "Absolute threshold used to filter detected peaks based on their raw intensity."
        )
        self.w_rel_threshold = widgets.Text(value=self.rel_threshold)
        self.rel_threshold = self._bold_widget("Relative threshold", self.w_rel_threshold)
        self.rel_threshold_def = self.create_help_text(
            "Relative threshold used to filter detected peaks based on their relative intensity."
        )
        self.w_sigma_ratio = widgets.Text(value=self.sigma_ratio)
        self.sigma_ratio = self._bold_widget("Sigma ratio", self.w_sigma_ratio)
        self.sigma_ratio_def = self.create_help_text(
            "DoG method parameter. The ratio between the standard deviation of Gaussian " \
            "Kernels used for computing the Difference of Gaussians."
        )
        self.w_min_sigma = widgets.Text(value=self.min_sigma)
        self.min_sigma = self._bold_widget("Minimum sigma", self.w_min_sigma)
        self.min_sigma_def = self.create_help_text(
            "Minimum sigma value used for blob detection."
        )
        self.w_max_sigma = widgets.Text(value=self.max_sigma)
        self.max_sigma = self._bold_widget("Maximum sigma", self.w_max_sigma)
        self.max_sigma_def = self.create_help_text(
            "Maximum sigma value used for blob detection."
        )

    def create_method_widgets(self):
        """Create method selection widgets."""
        label_method = widgets.HTML(value="<b>Peak Detection Method</b>")
        method_radio = widgets.RadioButtons(
            options=['persistent_homology', 'peak_local_max', 'LoG', 'DoG', 'DoH'],
            value='DoG',
            description='',
            disabled=False
        )
        method_widget = widgets.VBox([label_method, method_radio])
        
        label_mode = widgets.HTML(value="<b>Analysis Mode</b>")
        mode_radio = widgets.RadioButtons(
            options=['tic', 'mass_per_mass', '3D'],
            value='mass_per_mass',
            description='',
            disabled=False
        )
        mode_widget = widgets.VBox([label_mode, mode_radio])
    
        return method_widget, mode_widget, method_radio, mode_radio 

    def on_checkbox_change(self, change):
        if change['new'] is True:  # Mode automatique
            self._w_mod_time.layout.display = 'none'
            self.mod_time.layout.display = 'none'
        else:  # Mode manuel
            self._w_mod_time.layout.display = 'block'
            self.mod_time.layout.display = 'block'


    def _create_gcgcms_widgets(self):
        """Create GC×GC-MS specific widgets."""
        self.txt_title = widgets.HTML('<H1>Peak Detection</H1>')
        # Method and mode widgets
        self.w_method, self.w_mode, self.r_method, self.r_mode = self.create_method_widgets()
       # NIST matching
        self.nist = widgets.Checkbox(
            value=self.nist,
            description='Enable NIST Database Matching',
            style=self.style,
            disabled=False
        )

        self.mod_time_checkbox = widgets.Checkbox(
            value=True,
            description='Automatic detection for Modulation time',
            style=self.style,
            disabled=False
        ) 
        self._w_mod_time = widgets.Text(value=self.mod_time)
        self.mod_time = self._bold_widget("Manual modulation time", self._w_mod_time)

        self.plot = widgets.Checkbox(
            value=self.plot,
            description='Generate plots',
            style=self.style,
            disabled=False
        )
        # Toujours à True pour le moment
        self.is_area_deconvolution = widgets.Checkbox(
            # value=self.is_area_deconvolution,
            value=True,
            description='Deconvolution',
            style=self.style,
            disabled=True
        )

        # Action widgets
        self.run_button, self.stop_button, self.clear_button, self.output = self._create_action_widgets()
        self.mod_time_checkbox.observe(self.on_checkbox_change, names='value')
        self.on_checkbox_change({'new': self.mod_time_checkbox.value})

    def _validate_parameters(self):
        """Validate input parameters."""
        errors = []
        try:
            noise_val = float(self.w_noise_factor.value)
            if noise_val < 0:
                errors.append("The noise factor must be non-negative")
        except ValueError:
            errors.append("The noise factor must be a valid number")
        try:
            abs_val = float(self.w_abs_threshold.value)
            if abs_val < 0:
                errors.append("Absolute threshold must be non-negative")
        except ValueError:
            errors.append("Absolute threshold must be a valid number")
        try:
            rel_val = float(self.w_rel_threshold.value)
            if rel_val < 0 or rel_val > 1:
                errors.append("Relative threshold must be between 0 and 1")
        except ValueError:
            errors.append("Relative threshold must be a valid number")
        try:
            min_sigma_val = float(self.w_min_sigma.value)
            if min_sigma_val < 0:
                errors.append("Minimum sigma must be positive")
        except ValueError:
            errors.append("Minimum sigma must be a valid number")
        try:
            max_sigma_val = float(self.w_max_sigma.value)
            if max_sigma_val <= 0:
                errors.append("Maximum sigma must be positive")
            elif 'min_sigma_val' in locals() and max_sigma_val < min_sigma_val:
                errors.append("Maximum sigma must be greater than or equal to minimum sigma")
        except ValueError:
            errors.append("Maximum sigma must be a valid number")
        try:
            sigma_ratio_val = float(self.w_sigma_ratio.value)
            if sigma_ratio_val < 0:
                errors.append("Sigma ratio must be positive")
        except ValueError:
            errors.append("Sigma ratio must be a valid number")
        try:
            if not self.mod_time_checkbox.value:  # Only validate if manual mode is selected
                mod_time_val = float(self._w_mod_time.value)
                if mod_time_val < 0:
                    errors.append("Modulation time must be positive")
        except ValueError:
            errors.append("Modulation time must be a valid number")

        return errors

    def get_all_files_from_selections(self):
        """
        Retrieves all supported files from all selections.
        Automatically determines whether it's a file or a folder.
        """
        all_files = []
        processed_files = set()
        already_seen_files = set()

        for i, fc in enumerate(self._choosers):
            selected = fc.selected_path
            if not selected:
                continue

            try:
                selected_path = Path(selected)

                if fc.selected_filename:
                    # C'est un fichier spécifique
                    full_path = selected_path / fc.selected_filename
                    full_path_str = str(full_path)
                    
                    # Vérifier si ce fichier spécifique a déjà été traité
                    if full_path_str in processed_files:
                        self.output.append_stdout(f"⚠️  File already processed: {full_path_str}")
                        continue
                    
                    name_without_ext = fc.selected_filename.rsplit('.', 1)[0]

                    if fc.selected_filename.endswith(".cdf") and name_without_ext in already_seen_files:
                        self.output.append_stdout(f"⚠️  File already processed: {name_without_ext}.cdf")
                        continue

                    if fc.selected_filename.endswith(".h5"):
                        already_seen_files.add(name_without_ext)

                    if fc.selected_filename.endswith(self.supported_extensions):
                        all_files.append(full_path_str)
                        processed_files.add(full_path_str)  # Marquer ce fichier comme traité
                    else:
                        self.output.append_stdout(f"⚠️  Unsupported file ignored: {fc.selected_filename}")
                        self.output.append_stdout(f"   Supported extensions: {', '.join(self.supported_extensions)}")

                else:
                    # C'est un dossier
                    selected_path_str = str(selected_path)
                    if selected_path_str in processed_files:
                        self.output.append_stdout(f"⚠️  Folder already processed: {selected_path_str}")
                        continue
                    
                    processed_files.add(selected_path_str)
                    dir_files = self._get_files_from_directory(selected_path)

                    for f in dir_files:
                        path = str(Path(f))
                        if path not in processed_files:
                            all_files.append(path)
                            processed_files.add(path)

            except Exception as e:
                self.output.append_stdout(f"❌ Error while processing selection '{selected}': {e}")

        return all_files


    def _on_button_click(self, b):
        """Handle button click event to start analysis."""
        self.output.clear_output()
        self.output.append_stdout("🚀 Initializing GC×GC-MS analysis...\n")

        # Valider les paramètres
        errors = self._validate_parameters()
        if errors:
            self.output.append_stdout("❌ Parameter validation failed:")
            for error in errors:
                self.output.append_stdout(f"  • {error}")
            return
        
        if not hasattr(self, 'output_chooser') or not self.output_chooser.selected_path:
            self.output.append_stdout("Output directory cannot be empty")
            return
        
        # Obtenir tous les fichiers sélectionnés
        self.output.append_stdout("\n📂 Collecting files from selections...")
        selected_files = self.get_all_files_from_selections()
        
        if not selected_files:
            self.output.append_stdout("❌ No compatible files (.cdf or .h5) found in selections.")
            self.output.append_stdout("💡 Please select files or folders containing .cdf or .h5 files.")
            return

        self.output.append_stdout(f"\n✅ {len(selected_files)} compatible files found")

        self._start_subprocess_analysis(selected_files)
            

    def _start_subprocess_analysis(self, selected_files):
            """Start the analysis in a separate subprocess to keep the UI responsive."""
            self.output.append_stdout("\n" + "="*60 + "\n")
            self.output.append_stdout("🔄 Starting Peak Detection ...\n")

            analysis_params = [
                    sys.executable,
                    "/app/src/peak_detection_analyze_cli.py", #ds le docker
                    "--output", self.get_output_path(),
                    "--method", self.r_method.value,
                    "--mode", self.r_mode.value,
                    "--noise_factor", (self.w_noise_factor.value),
                    "--min_persistence", (self._min_persistence),
                    "--abs_threshold", (self.w_abs_threshold.value),
                    "--rel_threshold", (self.w_rel_threshold.value),
                    "--min_distance", self._min_distance,
                    "--min_sigma", self.w_min_sigma.value,
                    "--max_sigma", self.w_max_sigma.value,
                    "--sigma_ratio", self.w_sigma_ratio.value,
                    "--num_sigma", self._num_sigma,
                    "--match_factor_min", self._match_factor_min,
                    "--overlap", self._overlap,
                    "--eps", self._eps,
                    "--min_samples", self._min_samples,
                    "--mod_time", self._w_mod_time.value if not self.mod_time_checkbox.value else "0"
                ]
            #cas des booleens
            if self._cluster:
                analysis_params.append("--cluster")
            if self._formated_spectra:
                analysis_params.append("--formated_spectra")
            if self.nist.value:
                analysis_params.append("--nist")
            if self.plot.value:
                analysis_params.append("--plot")
            if self.is_area_deconvolution.value:
                analysis_params.append("--is_area_deconvolution")
            # cas des listes
            analysis_params += ["--input"] + selected_files

            analysis_params = list(map(str, analysis_params))

            # print("▶️ Lancement en tâche de fond :", " ".join(analysis_params))

            # self.stop_button.disabled = False
            # self.run_button.disabled = True 

            
            start_time = time.time()
            self.current_process = subprocess.Popen(
                analysis_params,
                stdout=subprocess.PIPE,
                stderr=subprocess.STDOUT,
                text=True,
                env={'PYTHONUNBUFFERED': '1'}
            )
            try:
                while self.current_process.poll() is None and (time.time() - start_time) < 600:
                    line = self.current_process.stdout.readline()
                    if line:
                        self.output.append_stdout(line)
                        start_time = time.time()  # Reset timeout
                    time.sleep(0.1)
                
                # Gestion de fin
                if self.current_process and self.current_process.poll() is None:
                    self.output.append_stdout("⏰ Analysis timed out\n")
                    self.current_process.terminate()
                    time.sleep(1)
                    if self.current_process.poll() is None:
                        self.current_process.kill()
                elif self.current_process:
                    retcode = self.current_process.returncode
                    if retcode == 0:
                        self.output.append_stdout("\n✅ Analysis completed successfully\n")
                    # elif retcode < 0: #killed by signal
                    #     self.output.append_stdout(f"\n🛑 Analysis was stopped by user\n")
                    else:
                        self.output.append_stdout(f"\n❌ Analysis failed with code {retcode}\n")

            except Exception as e:
                self.output.append_stdout(f"❌ Error: {e}\n")
            # finally:
            #     self.stop_button.disabled = True
            #     self.run_button.disabled = False
            #     self.current_process = None


    def display(self):
        """Display the complete UI."""
        display(
            self.txt_title,
            widgets.VBox([self._vbox, self._vbox2]),
            widgets.HBox([self.w_method, self.w_mode]),
            self.nist,
            self.plot,
            self.is_area_deconvolution,
            widgets.HBox([
                self.mod_time_checkbox,
                self.mod_time
            ]),
            self.noise_factor,
            self.noise_factor_def,
            self.abs_threshold,
            self.abs_threshold_def,
            self.rel_threshold,
            self.rel_threshold_def,
            self.min_sigma,
            self.min_sigma_def,
            self.max_sigma,
            self.max_sigma_def,
            self.sigma_ratio,
            self.sigma_ratio_def,
            widgets.HBox([self.run_button, self.stop_button, self.clear_button]),
            self.output
        )

In [ ]:
gcms_ui = GCGCMSAnalysisUI()
gcms_ui.display()